In [ ]:
import pandas as pd
from google.colab import drive


drive.mount('/content/drive')
path = '/content/drive/MyDrive/fastman_packages_1774463772436.xlsx'
df = pd.read_excel(path, dtype={'باركود': str})

cols_to_keep = [ 'باركود','إسم الزبون','الحالة','عدد محاولات السائق','إسم السائق','مدينة المرسل',
'المدينة', 'المنطقة', 'معرف العميل', 'طريقة الدفع', 'طريقة التحصيل', 'التحصيل', 'سعر التوصيل', 'تاريخ الحجز',
'تاريخ التوصيل المتوقع', 'تاريخ التوصيل','تاريخ الإرجاع','الوقت المستغرق','حالة المغلقة','سبب الفشل 1',
'سبب الفشل 2', 'سبب الفشل 3']

existing_cols = [col for col in cols_to_keep if col in df.columns]
df = df[existing_cols]

date_cols = [ 'تاريخ الحجز', 'تاريخ التوصيل المتوقع', 'تاريخ التوصيل', 'تاريخ الإرجاع']

for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')



# Categorical → fill with "Unknown"
cat_cols = df.select_dtypes(include='object').columns
for col in cat_cols:
    df[col] = df[col].fillna('Unknown')

# Numerical → fill with median
num_cols = df.select_dtypes(include=['int64', 'float64']).columns
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())



# Delivery Delay
df['Delivery_Delay'] = (df['تاريخ التوصيل'] - df['تاريخ الحجز']).dt.days

# Fill missing delay using expected date
df['Delivery_Delay'] = df['Delivery_Delay'].fillna(
    (df['تاريخ التوصيل المتوقع'] - df['تاريخ الحجز']).dt.days
)

# Late Flag
df['Is_Late'] = df['Delivery_Delay'].apply(lambda x: 1 if x > 0 else 0)

# Returned Flag
df['Is_Returned'] = df['تاريخ الإرجاع'].notna().astype(int)

# -------------------------------
# 7. City → Driver Mapping
# -------------------------------
city_driver_map = {"اربد": "الشمال مندوبين الشمال","الاغوار": "ابو جواد الاغوار","البلقاء": "معتصم السيد عبيد",
    "الرصيفة": "احمد الشرمان","الرمثا": "الشمال مندوبين الشمال","الزرقاء": "احمد الشرمان","السلط": "معتصم السيد عبيد",
   "الطفيلة": "معاذ دبور","العقبة": "معاذ دبور","الكرك": "معاذ دبور","المفرق": "ابو خليل الشمال","الموقر": "حمزة عشا",
    "جرش": "زيد الشمال","عجلون": "زيد الشمال","ابو علندا": "حمزة عشا","ابو نصير": "حسين العلاك","الاشرفية": "زايد قزاقزة","البنيات": "حمزة عشا",
    "البيادر": "رضوان المعايطة","الجاردنز": "يوسف نوفل", "الجبيهة": "حسين العلاك", "الجيزة": "صالح مادبا", "الرابية": "يوسف نوفل",
    "الزهور": "حمزة عشا","السابع": "يوسف نوفل", "الشميساني": "يوسف نوفل","الصويفية": "يوسف نوفل","العبدلي": "يوسف نوفل",
    "القويسمة": "حمزة عشا","اللويبدة": "يوسف نوفل","المدينة الرياضية": "زايد قزاقزة","المطار": "صالح مادبا","المقابلين": "حمزة عشا",
    "المنارة": "زايد قزاقزة","النزهة": "زايد قزاقزة","الهاشمي": "زايد قزاقزة","الوحدات": "يوسف نوفل","الياسمين": "حمزة عشا",
    "ام اذينة": "مهند سليم", "ام نوارة": "زايد قزاقزة", "تلاع العلي": "رضوان المعايطة", "جبل التاج": "زايد قزاقزة", "جبل الحسين": "حسين العلاك",
    "جبل النصر": "زايد قزاقزة","جبل عمان": "يوسف نوفل","حي نزال": "حمزة عشا","خريبة السوق": "حمزة عشا","خلدا": "معتصم السيد عبيد",
    "دابوق": "معتصم السيد عبيد","سحاب": "حمزة عشا","شفا بدران": "حسين العلاك","صويلح": "رضوان المعايطة","ضاحية الرشيد": "حسين العلاك",
    "طبربور": "زايد قزاقزة","عبدون": "يوسف نوفل","عمان": "حمزة عشا","ماركا الجنوبية": "زايد قزاقزة","ماركا الشمالية": "زايد قزاقزة",
    "مرج الحمام": "حمزة عشا","عين الباشا": "معتصم السيد عبيد","مادبا": "صالح مادبا","معان": "معاذ دبور"}

df['المدينة'] = df['المدينة'].astype(str).str.strip().str.lower()
city_driver_map = {k.strip().lower(): v for k, v in city_driver_map.items()}

mask = df['إسم السائق'] == "Unknown"
df.loc[mask, 'إسم السائق'] = df.loc[mask, 'المدينة'].map(city_driver_map)

df['إسم السائق'] = df['إسم السائق'].fillna("Unknown")

date_cols = ['تاريخ الحجز', 'تاريخ التوصيل المتوقع', 'تاريخ التوصيل']

for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')


df['Scheduled_Days'] = (df['تاريخ التوصيل المتوقع'] - df['تاريخ الحجز']).dt.days
df['Actual_Days'] = (df['تاريخ التوصيل'] - df['تاريخ الحجز']).dt.days
df['Scheduled_Days'] = df['Scheduled_Days'].fillna(df['Scheduled_Days'].median())
df['Actual_Days'] = df['Actual_Days'].fillna(df['Actual_Days'].median())


df.to_excel("final_cleaned.xlsx", index=False)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
